In [1]:
import sys
import json
import os
import re
import time
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: e:\slde-aft


In [2]:
sample_path = project_root / "data" / "carb_dev_sample.jsonl"
prompt_path = project_root / "prompts" / "openie_carb_v1.txt"

with prompt_path.open("r", encoding="utf-8") as f:
    prompt_template = f.read()

records = []
with sample_path.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

smoke_records = records[:3]

print("Total CaRB subset records:", len(records))
print("Smoke-test records:", len(smoke_records))

for record in smoke_records:
    print("\nID:", record["sentence_id"])
    print("Sentence:", record["sentence"])
    print("Gold triples:", record["gold_triples"])

Total CaRB subset records: 30
Smoke-test records: 3

ID: carb_test_0001
Sentence: 32.7 % of all households were made up of individuals and 15.7 % had someone living alone who was 65 years of age or older .
Gold triples: [{'subject': '32.7 % of all households', 'predicate': 'were made up of', 'object': 'individuals'}, {'subject': '15.7 % of all households', 'predicate': 'had', 'object': 'someone living alone who was 65 years of age or older'}]

ID: carb_test_0002
Sentence: A CEN forms an important but small part of a Local Strategic Partnership .
Gold triples: [{'subject': 'A CEN', 'predicate': 'forms', 'object': 'an important part of a Local Strategic Partnership'}, {'subject': 'A CEN', 'predicate': 'forms', 'object': 'a small part of a Local Strategic Partnership'}, {'subject': 'a Strategic Partnership', 'predicate': 'is', 'object': 'Local'}]

ID: carb_test_0003
Sentence: A Democrat , he became the youngest mayor in Pittsburgh 's history in September 2006 at the age of 26 .
Gold tripl

In [3]:
test_prompt = prompt_template.replace(
    "{SENTENCE}",
    smoke_records[0]["sentence"]
)

print(test_prompt)

You are an Open Information Extraction system.

Extract factual information explicitly stated in the sentence.

Return every extraction as a subject-predicate-object triple.

Rules:
- Use the exact or near-exact words from the sentence where possible.
- Use natural-language relation phrases for predicates.
- Do not invent facts.
- Do not provide explanations.
- Return only a valid JSON array.
- If no factual triple can be extracted, return [].

Required JSON format:
[
  {
    "subject": "text",
    "predicate": "relation phrase",
    "object": "text"
  }
]

Sentence:
32.7 % of all households were made up of individuals and 15.7 % had someone living alone who was 65 years of age or older .


In [ ]:
import os
from getpass import getpass

API_KEY = getpass("Enter OpenRouter API key:openrouter_key").strip()

if not API_KEY:
    raise RuntimeError("No API key entered.")

print("API key loaded successfully.")

API key loaded successfully.


In [5]:
import json
import re
import time
import requests

MODEL_NAME = "openrouter/auto"

def parse_json_array(text: str):
    text = text.strip()

    match = re.search(r"\[.*\]", text, flags=re.DOTALL)
    if not match:
        return [], "No JSON array found in model response."

    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError as exc:
        return [], f"Invalid JSON: {exc}"

    if not isinstance(parsed, list):
        return [], "Parsed response is not a JSON list."

    valid_triples = []

    for item in parsed:
        if not isinstance(item, dict):
            continue

        required_keys = {"subject", "predicate", "object"}
        if not required_keys.issubset(item):
            continue

        subject = str(item["subject"]).strip()
        predicate = str(item["predicate"]).strip()
        object_ = str(item["object"]).strip()

        if subject and predicate and object_:
            valid_triples.append(
                {
                    "subject": subject,
                    "predicate": predicate,
                    "object": object_,
                }
            )

    return valid_triples, None


def extract_carb_openie(sentence: str, prompt_template: str):
    prompt = prompt_template.replace("{SENTENCE}", sentence)

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-Title": "SLDE-AFT-CaRB-Smoke-Test",
    }

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": prompt,
            }
        ],
        "temperature": 0.0,
    }

    start_time = time.perf_counter()

    try:
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers=headers,
            json=payload,
            timeout=90,
        )
        response.raise_for_status()

        response_json = response.json()
        raw_response = response_json["choices"][0]["message"]["content"]

        triples, parse_error = parse_json_array(raw_response)
        latency_seconds = time.perf_counter() - start_time

        return {
            "raw_response": raw_response,
            "triples": triples,
            "parse_error": parse_error,
            "api_error": None,
            "latency_seconds": latency_seconds,
        }

    except Exception as exc:
        latency_seconds = time.perf_counter() - start_time

        return {
            "raw_response": None,
            "triples": [],
            "parse_error": None,
            "api_error": str(exc),
            "latency_seconds": latency_seconds,
        }


print("CaRB OpenIE extractor is ready.")

CaRB OpenIE extractor is ready.


In [7]:
import json
from pathlib import Path

project_root = Path.cwd().parent

sample_path = project_root / "data" / "carb_dev_sample.jsonl"
prompt_path = project_root / "prompts" / "openie_carb_v1.txt"

with prompt_path.open("r", encoding="utf-8") as f:
    prompt_template = f.read()

records = []

with sample_path.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if line:
            records.append(json.loads(line))

smoke_records = records[:3]

print("Total CaRB subset records:", len(records))
print("Smoke-test records:", len(smoke_records))

Total CaRB subset records: 30
Smoke-test records: 3


In [8]:
test_record = smoke_records[0]

result = extract_carb_openie(
    sentence=test_record["sentence"],
    prompt_template=prompt_template,
)

print("Sentence:")
print(test_record["sentence"])

print("\nGold triples:")
for triple in test_record["gold_triples"]:
    print(triple)

print("\nPredicted triples:")
print(result["triples"])

print("\nRaw response:")
print(result["raw_response"])

print("\nParse error:", result["parse_error"])
print("API error:", result["api_error"])
print("Latency (seconds):", round(result["latency_seconds"], 2))

Sentence:
32.7 % of all households were made up of individuals and 15.7 % had someone living alone who was 65 years of age or older .

Gold triples:
{'subject': '32.7 % of all households', 'predicate': 'were made up of', 'object': 'individuals'}
{'subject': '15.7 % of all households', 'predicate': 'had', 'object': 'someone living alone who was 65 years of age or older'}

Predicted triples:
[{'subject': '32.7% of all households', 'predicate': 'were made up of', 'object': 'individuals'}, {'subject': '15.7% of all households', 'predicate': 'had', 'object': 'someone living alone who was 65 years of age or older'}]

Raw response:
[
  {
    "subject": "32.7% of all households",
    "predicate": "were made up of",
    "object": "individuals"
  },
  {
    "subject": "15.7% of all households",
    "predicate": "had",
    "object": "someone living alone who was 65 years of age or older"
  }
]

Parse error: None
API error: None
Latency (seconds): 4.19


In [9]:
predictions = []

for record in smoke_records:
    result = extract_carb_openie(
        sentence=record["sentence"],
        prompt_template=prompt_template,
    )

    predictions.append(
        {
            "sentence_id": record["sentence_id"],
            "sentence": record["sentence"],
            "gold_triples": record["gold_triples"],
            "predicted_triples": result["triples"],
            "raw_response": result["raw_response"],
            "parse_error": result["parse_error"],
            "api_error": result["api_error"],
            "latency_seconds": result["latency_seconds"],
            "model": MODEL_NAME,
            "prompt_version": "openie_carb_v1",
        }
    )

    print(record["sentence_id"])
    print("Predictions:", result["triples"])
    print("Parse error:", result["parse_error"])
    print("API error:", result["api_error"])
    print("Latency:", round(result["latency_seconds"], 2), "seconds")
    print("-" * 80)

carb_test_0001
Predictions: [{'subject': '32.7% of all households', 'predicate': 'were made up of', 'object': 'individuals'}, {'subject': '15.7% of all households', 'predicate': 'had', 'object': 'someone living alone who was 65 years of age or older'}]
Parse error: None
API error: None
Latency: 3.9 seconds
--------------------------------------------------------------------------------
carb_test_0002
Predictions: [{'subject': 'A CEN', 'predicate': 'forms', 'object': 'an important but small part of a Local Strategic Partnership'}]
Parse error: None
API error: None
Latency: 3.76 seconds
--------------------------------------------------------------------------------
carb_test_0003
Predictions: [{'subject': 'he', 'predicate': 'is', 'object': 'a Democrat'}, {'subject': 'he', 'predicate': "became the youngest mayor in Pittsburgh's history", 'object': 'in September 2006'}, {'subject': 'he', 'predicate': "became the youngest mayor in Pittsburgh's history", 'object': 'at the age of 26'}]
Parse

In [10]:
from pathlib import Path
import json

output_dir = project_root / "outputs" / "carb_smoke_test_3"
output_dir.mkdir(parents=True, exist_ok=True)

predictions_path = output_dir / "predictions.json"

with predictions_path.open("w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

config = {
    "experiment_id": "carb_smoke_test_3",
    "decision": "DEC-001",
    "dataset": "CaRB",
    "gold_source": "data/CaRB/data/gold/test.tsv",
    "subset_source": "data/carb_dev_sample.jsonl",
    "subset_size": 3,
    "model": MODEL_NAME,
    "prompt_file": "prompts/openie_carb_v1.txt",
    "prompt_version": "openie_carb_v1",
    "temperature": 0.0,
    "purpose": "Verify CaRB OpenIE extraction, JSON parsing, raw-response saving, and latency recording.",
}

config_path = output_dir / "config.json"

with config_path.open("w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Saved predictions to:", predictions_path)
print("Saved configuration to:", config_path)

Saved predictions to: e:\slde-aft\outputs\carb_smoke_test_3\predictions.json
Saved configuration to: e:\slde-aft\outputs\carb_smoke_test_3\config.json


In [13]:
import sys
from pathlib import Path

# Your notebook runs from E:\slde-aft\notebooks
project_root = Path.cwd().parent

# Add E:\slde-aft so Python can find the src folder
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Notebook folder:", Path.cwd())
print("Project root:", project_root)
print("src exists:", (project_root / "src").exists())

Notebook folder: e:\slde-aft\notebooks
Project root: e:\slde-aft
src exists: True


In [14]:
from src.evaluator import Triple, compute_precision_recall_f1

print("Evaluator imported successfully.")

Evaluator imported successfully.


In [15]:
from src.evaluator import Triple, compute_precision_recall_f1
import json

predictions_path = project_root / "outputs" / "carb_smoke_test_3" / "predictions.json"

with predictions_path.open("r", encoding="utf-8") as f:
    saved_predictions = json.load(f)

gold_triples = []
predicted_triples = []

for record in saved_predictions:
    for triple in record["gold_triples"]:
        gold_triples.append(
            Triple(
                subject=triple["subject"],
                predicate=triple["predicate"],
                object=triple["object"],
            )
        )

    for triple in record["predicted_triples"]:
        predicted_triples.append(
            Triple(
                subject=triple["subject"],
                predicate=triple["predicate"],
                object=triple["object"],
            )
        )

metrics = compute_precision_recall_f1(
    gold_triples=gold_triples,
    predicted_triples=predicted_triples,
)

print("Internal CaRB smoke-test metrics")
for name, value in metrics.items():
    if isinstance(value, float):
        print(f"{name}: {value:.4f}")
    else:
        print(f"{name}: {value}")

Internal CaRB smoke-test metrics
precision: 0.3333
recall: 0.1818
f1: 0.2353
true_positives: 2
false_positives: 4
false_negatives: 9
gold_count: 11
prediction_count: 6


In [16]:
import json

metrics_path = project_root / "outputs" / "carb_smoke_test_3" / "metrics_internal.json"

with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Saved metrics to:", metrics_path)

Saved metrics to: e:\slde-aft\outputs\carb_smoke_test_3\metrics_internal.json


In [17]:
import json
from src.evaluator import Triple, normalize_triple

predictions_path = project_root / "outputs" / "carb_smoke_test_3" / "predictions.json"

with predictions_path.open("r", encoding="utf-8") as f:
    saved_predictions = json.load(f)

for record in saved_predictions:
    print("Sentence:", record["sentence"])
    print()

    gold_set = {normalize_triple(Triple(**t)) for t in record["gold_triples"]}
    pred_set = {normalize_triple(Triple(**t)) for t in record["predicted_triples"]}

    true_positives = gold_set & pred_set
    false_positives = pred_set - gold_set
    false_negatives = gold_set - pred_set

    print("True positives:")
    for t in sorted(true_positives, key=lambda x: (x.subject, x.predicate, x.object)):
        print(f"  S: {t.subject!r}")
        print(f"  P: {t.predicate!r}")
        print(f"  O: {t.object!r}")
        print()

    print("False positives (predicted but not in gold):")
    for t in sorted(false_positives, key=lambda x: (x.subject, x.predicate, x.object)):
        print(f"  S: {t.subject!r}")
        print(f"  P: {t.predicate!r}")
        print(f"  O: {t.object!r}")
        print()

    print("False negatives (in gold but not predicted):")
    for t in sorted(false_negatives, key=lambda x: (x.subject, x.predicate, x.object)):
        print(f"  S: {t.subject!r}")
        print(f"  P: {t.predicate!r}")
        print(f"  O: {t.object!r}")
        print()

    print("-" * 80)
    print()

Sentence: 32.7 % of all households were made up of individuals and 15.7 % had someone living alone who was 65 years of age or older .

True positives:
  S: '15.7% of all households'
  P: 'had'
  O: 'someone living alone who was 65 years of age or older'

  S: '32.7% of all households'
  P: 'were made up of'
  O: 'individuals'

False positives (predicted but not in gold):
False negatives (in gold but not predicted):
--------------------------------------------------------------------------------

Sentence: A CEN forms an important but small part of a Local Strategic Partnership .

True positives:
False positives (predicted but not in gold):
  S: 'a cen'
  P: 'forms'
  O: 'an important but small part of a local strategic partnership'

False negatives (in gold but not predicted):
  S: 'a cen'
  P: 'forms'
  O: 'a small part of a local strategic partnership'

  S: 'a cen'
  P: 'forms'
  O: 'an important part of a local strategic partnership'

  S: 'a strategic partnership'
  P: 'is'
  O: '

In [18]:
import json
from pathlib import Path

# Reuse project_root from earlier; if not defined, re-run the path-setup cell:
# import sys
# from pathlib import Path
# project_root = Path.cwd().parent
# if str(project_root) not in sys.path:
#     sys.path.insert(0, str(project_root))

subset_path = project_root / "data" / "carb_dev_sample.jsonl"
output_dir = project_root / "outputs" / "carb_10_dev"
output_dir.mkdir(parents=True, exist_ok=True)

# Load all available sentences
with subset_path.open("r", encoding="utf-8") as f:
    all_records = [json.loads(line) for line in f if line.strip()]

print("Total sentences available in carb_dev_sample.jsonl:", len(all_records))

# Take the first 10
subset_10 = all_records[:10]
print("Using subset size:", len(subset_10))

# Save the subset definition for reproducibility
subset_file = output_dir / "subset_10.jsonl"
with subset_file.open("w", encoding="utf-8") as f:
    for record in subset_10:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Saved subset to:", subset_file)

Total sentences available in carb_dev_sample.jsonl: 30
Using subset size: 10
Saved subset to: e:\slde-aft\outputs\carb_10_dev\subset_10.jsonl


In [20]:
!pip install openai -q

In [ ]:
import os

# Replace THIS_WITH_YOUR_OPENROUTER_API_KEY with your actual key
os.environ["OPENROUTER_API_KEY"] = "api_key_inseret"

print("API key set:", "OPENROUTER_API_KEY" in os.environ)

API key set: True


In [27]:
import os
import time
import json
from pathlib import Path
from openai import OpenAI

# Reuse project_root if needed
# import sys
# from pathlib import Path
# project_root = Path.cwd().parent
# if str(project_root) not in sys.path:
#     sys.path.insert(0, str(project_root))

# Load the 10-sentence subset
subset_file = project_root / "outputs" / "carb_10_dev" / "subset_10.jsonl"
with subset_file.open("r", encoding="utf-8") as f:
    subset_10 = [json.loads(line) for line in f if line.strip()]

# Load prompt
prompt_path = project_root / "prompts" / "openie_carb_v1.txt"
with prompt_path.open("r", encoding="utf-8") as f:
    base_prompt = f.read()

# Initialize OpenRouter client
client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

MODEL_NAME = "openrouter/auto"
TEMPERATURE = 0.0

predictions = []
latencies = []

for i, record in enumerate(subset_10, start=1):
    sentence = record["sentence"]
    gold_triples = record["gold_triples"]

    prompt = base_prompt.replace("{SENTENCE}", sentence)

    start = time.perf_counter()
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
    )
    elapsed = time.perf_counter() - start
    latencies.append(elapsed)

    raw_text = response.choices[0].message.content.strip()

    # Parse JSON
    try:
        parsed = json.loads(raw_text)
        if isinstance(parsed, dict) and "triples" in parsed:
            parsed_triples = parsed["triples"]
        elif isinstance(parsed, list):
            parsed_triples = parsed
        else:
            raise ValueError("Unexpected JSON shape")
    except Exception as e:
        print(f"Sentence {i} parse error:", e)
        print("Raw response:", raw_text)
        parsed_triples = []

    pred_record = {
        "sentence_id": record.get("sentence_id", i),
        "sentence": sentence,
        "gold_triples": gold_triples,
        "predicted_triples": parsed_triples,
        "raw_response": raw_text,
        "latency_seconds": elapsed,
    }

    predictions.append(pred_record)
    print(f"Sentence {i:2d} | latency: {elapsed:6.2f}s | predicted triples: {len(parsed_triples)}")

# Save predictions
output_dir = project_root / "outputs" / "carb_10_dev"
predictions_path = output_dir / "predictions.json"

with predictions_path.open("w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print()
print("Saved predictions to:", predictions_path)
print("Average latency:", sum(latencies) / len(latencies), "seconds")

Sentence  1 | latency:   4.17s | predicted triples: 2
Sentence  2 | latency:   3.81s | predicted triples: 1
Sentence  3 | latency:  13.35s | predicted triples: 4
Sentence  4 | latency:   2.71s | predicted triples: 3
Sentence  5 | latency:  19.81s | predicted triples: 4
Sentence  6 | latency:   4.58s | predicted triples: 3
Sentence  7 | latency:  10.82s | predicted triples: 1
Sentence  8 | latency:   8.26s | predicted triples: 3
Sentence  9 | latency:   3.72s | predicted triples: 2
Sentence 10 | latency:   5.44s | predicted triples: 2

Saved predictions to: e:\slde-aft\outputs\carb_10_dev\predictions.json
Average latency: 7.6669432999973655 seconds


In [28]:
import json
from src.evaluator import Triple, compute_precision_recall_f1

predictions_path = project_root / "outputs" / "carb_10_dev" / "predictions.json"

with predictions_path.open("r", encoding="utf-8") as f:
    saved_predictions = json.load(f)

gold_triples = []
predicted_triples = []

for record in saved_predictions:
    for triple in record["gold_triples"]:
        gold_triples.append(
            Triple(
                subject=triple["subject"],
                predicate=triple["predicate"],
                object=triple["object"],
            )
        )

    for triple in record["predicted_triples"]:
        predicted_triples.append(
            Triple(
                subject=triple["subject"],
                predicate=triple["predicate"],
                object=triple["object"],
            )
        )

metrics = compute_precision_recall_f1(
    gold_triples=gold_triples,
    predicted_triples=predicted_triples,
)

print("Internal CaRB 10-sentence metrics")
for name, value in metrics.items():
    if isinstance(value, float):
        print(f"{name}: {value:.4f}")
    else:
        print(f"{name}: {value}")

# Save metrics
metrics_path = project_root / "outputs" / "carb_10_dev" / "metrics_internal.json"
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print()
print("Saved metrics to:", metrics_path)

Internal CaRB 10-sentence metrics
precision: 0.1600
recall: 0.1143
f1: 0.1333
true_positives: 4
false_positives: 21
false_negatives: 31
gold_count: 35
prediction_count: 25

Saved metrics to: e:\slde-aft\outputs\carb_10_dev\metrics_internal.json


In [29]:
import json
from src.evaluator import Triple, normalize_triple

predictions_path = project_root / "outputs" / "carb_10_dev" / "predictions.json"

with predictions_path.open("r", encoding="utf-8") as f:
    saved_predictions = json.load(f)

tp_examples = []
fp_examples = []
fn_examples = []

for record in saved_predictions:
    gold_set = {normalize_triple(Triple(**t)) for t in record["gold_triples"]}
    pred_set = {normalize_triple(Triple(**t)) for t in record["predicted_triples"]}

    true_positives = gold_set & pred_set
    false_positives = pred_set - gold_set
    false_negatives = gold_set - pred_set

    for t in true_positives:
        tp_examples.append((record["sentence"], t, "TP"))
    for t in false_positives:
        fp_examples.append((record["sentence"], t, "FP"))
    for t in false_negatives:
        fn_examples.append((record["sentence"], t, "FN"))

print("Example TRUE POSITIVES:")
for sent, t, _ in tp_examples[:3]:
    print("Sentence:", sent)
    print(f"  S: {t.subject!r}")
    print(f"  P: {t.predicate!r}")
    print(f"  O: {t.object!r}")
    print()

print("Example FALSE POSITIVES:")
for sent, t, _ in fp_examples[:3]:
    print("Sentence:", sent)
    print(f"  S: {t.subject!r}")
    print(f"  P: {t.predicate!r}")
    print(f"  O: {t.object!r}")
    print()

print("Example FALSE NEGATIVES:")
for sent, t, _ in fn_examples[:3]:
    print("Sentence:", sent)
    print(f"  S: {t.subject!r}")
    print(f"  P: {t.predicate!r}")
    print(f"  O: {t.object!r}")
    print()

Example TRUE POSITIVES:
Sentence: 32.7 % of all households were made up of individuals and 15.7 % had someone living alone who was 65 years of age or older .
  S: '15.7% of all households'
  P: 'had'
  O: 'someone living alone who was 65 years of age or older'

Sentence: 32.7 % of all households were made up of individuals and 15.7 % had someone living alone who was 65 years of age or older .
  S: '32.7% of all households'
  P: 'were made up of'
  O: 'individuals'

Sentence: A cooling center is a temporary air-conditioned public space set up by local authorities to deal with the health effects of a heat wave .
  S: 'a cooling center'
  P: 'is set up by'
  O: 'local authorities'

Example FALSE POSITIVES:
Sentence: A CEN forms an important but small part of a Local Strategic Partnership .
  S: 'a cen'
  P: 'forms'
  O: 'an important but small part of a local strategic partnership'

Sentence: A Democrat , he became the youngest mayor in Pittsburgh 's history in September 2006 at the age o